In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

ModuleNotFoundError: No module named 'seaborn'

In [3]:
# Cargar el CSV completo
df = pd.read_csv('BTC USD 1 Min Data.csv')


print('Dataset cargado exitosamente')
print(f'  Filas    : {len(df):,}')
print(f'  Columnas : {df.shape[1]}')
print(f'  Memoria  : {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')

Dataset cargado exitosamente
  Filas    : 7,608,981
  Columnas : 6
  Memoria  : 348.3 MB


In [4]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 7608981 entries, 0 to 7608980
Data columns (total 6 columns):
 #   Column     Dtype  
---  ------     -----  
 0   Timestamp  int64  
 1   Open       float64
 2   High       float64
 3   Low        float64
 4   Close      float64
 5   Volume     float64
dtypes: float64(5), int64(1)
memory usage: 348.3 MB


In [6]:
df.describe().round(2)

,Timestamp,Open,High,Low,Close,Volume
count,7.608981e+06,7608981.00,7608981.00,7608981.00,7608981.00,7608981.00
mean,1.553645e+09,23385.86,23394.03,23377.52,23385.88,5.00
std,1.317914e+08,31297.13,31305.53,31288.69,31297.16,21.67
min,1.325376e+09,3.80,3.80,3.80,3.80,0.00
25%,1.439511e+09,455.23,455.39,455.05,455.23,0.02
50%,1.553645e+09,7970.01,7975.06,7965.06,7969.95,0.45
75%,1.667780e+09,37866.00,37886.72,37846.00,37866.34,2.80
max,1.781915e+09,126202.00,126272.00,126158.00,126202.00,5853.85


In [7]:
nulos = df.isnull().sum()
pct_nulos = (nulos / len(df) * 100).round(2)

resumen_nulos = pd.DataFrame({
    'Valores nulos': nulos,
    '% del total': pct_nulos
})

print(resumen_nulos)
print(f'\nTotal filas con al menos 1 nulo: {df.isnull().any(axis=1).sum():,}')

           Valores nulos  % del total
Timestamp              0          0.0
Open                   0          0.0
High                   0          0.0
Low                    0          0.0
Close                  0          0.0
Volume                 0          0.0

Total filas con al menos 1 nulo: 0


In [8]:
duplicados = df.duplicated().sum()
print(f'Filas duplicadas: {duplicados:,}')

Filas duplicadas: 0


In [9]:
df_clean = df.copy()

# 1. Eliminar filas donde Close es nulo (sin datos de precio)
df_clean = df_clean.dropna(subset=['Close'])

# 2. Eliminar filas con volumen = 0 Y todos los precios iguales (sin actividad real)
sin_actividad = (df_clean['Volume'] == 0) & (df_clean['Open'] == df_clean['Close'])
df_clean = df_clean[~sin_actividad]

# 3. Convertir Timestamp Unix a datetime
df_clean['Date'] = pd.to_datetime(df_clean['Timestamp'], unit='s')

# 4. Eliminar duplicados en Timestamp
df_clean = df_clean.drop_duplicates(subset=['Timestamp'])

# 5. Verificar coherencia: High >= Close >= Low
inconsistentes = df_clean[df_clean['High'] < df_clean['Low']]
print(f'Filas con High < Low (inconsistentes): {len(inconsistentes)}')

# Resumen
print(f'\nFilas originales : {len(df):,}')
print(f'Filas limpias    : {len(df_clean):,}')
print(f'Filas eliminadas : {len(df) - len(df_clean):,}')
print(f'\nPeríodo: {df_clean["Date"].min().date()} → {df_clean["Date"].max().date()}')


Filas con High < Low (inconsistentes): 0

Filas originales : 7,608,981
Filas limpias    : 6,296,619
Filas eliminadas : 1,312,362

Período: 2012-01-01 → 2026-06-20


In [ ]:
# Establecer Date como índice para poder hacer resample
df_idx = df_clean.set_index('Date')

# Resample a diario con reglas OHLCV estándar
df_daily = df_idx.resample('D').agg({
    'Open'  : 'first',
    'High'  : 'max',
    'Low'   : 'min',
    'Close' : 'last',
    'Volume': 'sum'
}).dropna(subset=['Close'])

df_daily = df_daily.reset_index()

print(f'Filas diarias: {len(df_daily):,}')
print(f'Período      : {df_daily["Date"].min().date()} → {df_daily["Date"].max().date()}')
print(f'Memoria      : {df_daily.memory_usage(deep=True).sum() / 1024:.1f} KB')
df_daily.head()

In [ ]:
# Liberar la memoria del dataframe original (ya no lo necesitamos)
del df, df_clean, df_idx
print('Memoria liberada')

In [ ]:
df_daily['Year'] = df_daily['Date'].dt.year

precio_anual = df_daily.groupby('Year')['Close'].agg(
    Promedio='mean',
    Maximo='max',
    Minimo='min',
    Cierre_Final='last'
).round(2)

# Calcular crecimiento año a año
precio_anual['Crecimiento_%'] = precio_anual['Promedio'].pct_change().mul(100).round(1)

print('Precio de Bitcoin por Año (USD):')
print(precio_anual.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

colores = ['#e74c3c' if x < 0 else '#2ecc71' for x in precio_anual['Crecimiento_%'].fillna(0)]

bars = ax.bar(precio_anual.index, precio_anual['Promedio'], color='#F7931A', alpha=0.85, edgecolor='white')

# Etiquetas de valor sobre cada barra
for bar, val in zip(bars, precio_anual['Promedio']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
            f'${val:,.0f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_title('Precio Promedio Anual de Bitcoin (USD)', fontsize=14, fontweight='bold')
ax.set_xlabel('Año')
ax.set_ylabel('Precio Promedio (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.grid(axis='y', alpha=0.3)
ax.set_xticks(precio_anual.index)

plt.tight_layout()
plt.savefig('analisis1_precio_anual.png', bbox_inches='tight')
plt.show()
print('Gráfica guardada: analisis1_precio_anual.png')